# Export Datawrapper files

Reads `source/precinct_results.csv`, `source/census_precinct.csv`, `source/citywide_trend.csv`, and `source/precincts.geojson`. Writes:

- `datawrapper/precincts.csv` — one row per precinct, for the choropleth
- `datawrapper/districts.csv` — one row per supervisor district, for bar charts
- `datawrapper/precincts.geojson` — the shapes Datawrapper joins to the precinct CSV
- `datawrapper/trend.csv` — Measures B, I, and D on each summary report, for a line chart
- `datawrapper/late_count.csv` — Measures B, I, D, and G, split into votes counted by November 4 and since, for a bar chart

Percentages are on a 0–100 scale, one decimal place. A rate is left blank when the precinct counted fewer than 50 ballots. `chan_pct` is also blank outside CA-11, and `turnout_pct` is blank where a precinct reports more ballots than registered voters.

Open this notebook from the folder after you upload it to Drive. It does not download anything and it does not upload anything.


In [ ]:
from pathlib import Path
import json

import pandas as pd

FOLDER_NAME = "sf-2026-drive"
MIN_BALLOTS = 50


def find_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *list(here.parents)[:4]]
    for candidate in candidates:
        if (candidate / "source" / "precinct_results.csv").exists():
            return candidate
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        base = Path("/content/drive/MyDrive")
        direct = base / FOLDER_NAME
        if (direct / "source" / "precinct_results.csv").exists():
            return direct
        matches = [
            path for path in base.rglob("precinct_results.csv") if path.parent.name == "source"
        ]
        if len(matches) == 1:
            return matches[0].parent.parent
    except ImportError:
        pass
    raise FileNotFoundError(
        "Could not find source/precinct_results.csv. "
        "Upload this folder to My Drive and set FOLDER_NAME to the folder name."
    )


ROOT = find_root()
SOURCE = ROOT / "source"
OUT = ROOT / "datawrapper"
OUT.mkdir(parents=True, exist_ok=True)
print(ROOT)


In [ ]:
votes = pd.read_csv(SOURCE / "precinct_results.csv", dtype={"precinct_id": str})
census = pd.read_csv(SOURCE / "census_precinct.csv", dtype={"precinct_id": str})
frame = votes.merge(census, on="precinct_id", how="left")

enough = frame["ballots"].fillna(0) >= MIN_BALLOTS
in_ca11 = frame["Congressional"] == 11


def rate(numerator, denominator, keep):
    share = numerator / denominator.where(denominator > 0)
    return (share * 100).round(1).where(keep)


def as_int(column):
    return frame[column].fillna(0).round().astype(int)


def readable(name):
    words = name.title().replace("/Usf", "/USF").replace("Mclaren", "McLaren").split(" ")
    return " ".join(w.lower() if w in ("Of",) else w for w in words)


ca11_status = pd.Series("In CA-11", index=frame.index)
ca11_status[in_ca11 & ~enough] = "In CA-11, fewer than 50 ballots"
ca11_status[~in_ca11] = "In CA-15"


precincts = pd.DataFrame({
    "precinct_id": frame["precinct_id"].astype(int),
    "precinct_name": frame["VotingPrecinctName"],
    "neighborhood": frame["Neighborhood"],
    "neighborhood_name": frame["Neighborhood"].map(readable),
    "supervisor_district": frame["Supervisorial"].astype(int),
    "congressional_district": frame["Congressional"].astype(int),
    "assembly_district": frame["Assembly"].astype(int),
    "registered_voters": as_int("registered"),
    "ballots_counted": as_int("ballots"),
    "mail_ballots": as_int("mail_ballots"),
    "turnout_pct": rate(frame["ballots"], frame["registered"], enough & (frame["ballots"] <= frame["registered"])),
    "mail_pct": rate(frame["mail_ballots"], frame["ballots"], enough),
    "hilton_votes": as_int("hilton"),
    "becerra_votes": as_int("becerra"),
    "hilton_pct": rate(frame["hilton"], frame["hilton"] + frame["becerra"], enough),
    "chan_votes": as_int("chan"),
    "wiener_votes": as_int("wiener"),
    "chan_pct": rate(frame["chan"], frame["chan"] + frame["wiener"], enough & in_ca11),
    "ca11_status": ca11_status,
    "measure_b_yes": as_int("measure_b__yes"),
    "measure_b_no": as_int("measure_b__no"),
    "measure_b_yes_pct": rate(
        frame["measure_b__yes"], frame["measure_b__yes"] + frame["measure_b__no"], enough
    ),
    "measure_d_yes": as_int("measure_d__yes"),
    "measure_d_no": as_int("measure_d__no"),
    "measure_d_yes_pct": rate(
        frame["measure_d__yes"], frame["measure_d__yes"] + frame["measure_d__no"], enough
    ),
    "measure_g_yes": as_int("measure_g__yes"),
    "measure_g_no": as_int("measure_g__no"),
    "measure_g_yes_pct": rate(
        frame["measure_g__yes"], frame["measure_g__yes"] + frame["measure_g__no"], enough
    ),
    "measure_i_yes": as_int("measure_i__yes"),
    "measure_i_no": as_int("measure_i__no"),
    "measure_i_yes_pct": rate(
        frame["measure_i__yes"], frame["measure_i__yes"] + frame["measure_i__no"], enough
    ),
    "pop_2020": frame["pop"].round().astype("Int64"),
    "hispanic_pct": rate(frame["hispanic"], frame["pop"], frame["pop"] > 0),
    "nh_white_pct": rate(frame["nh_white"], frame["pop"], frame["pop"] > 0),
    "nh_black_pct": rate(frame["nh_black"], frame["pop"], frame["pop"] > 0),
    "nh_asian_pct": rate(frame["nh_asian"], frame["pop"], frame["pop"] > 0),
    "bachelors_pct": rate(frame["ba_plus"], frame["b15003_e001"], frame["b15003_e001"] > 0),
    "renter_pct": rate(frame["b25003_e003"], frame["b25003_e001"], frame["b25003_e001"] > 0),
    "hh_150k_pct": rate(frame["hh_150k"], frame["b19001_e001"], frame["b19001_e001"] > 0),
    "foreign_born_pct": rate(frame["foreign_born"], frame["b05001_e001"], frame["b05001_e001"] > 0),
    "group_quarters_pct": rate(frame["gq"], frame["pop"], frame["pop"] > 0),
})
precincts = precincts.sort_values("precinct_id")
precincts.to_csv(OUT / "precincts.csv", index=False)
print(len(precincts), "precincts ->", OUT / "precincts.csv")


In [ ]:
def weighted(part, numerator, denominator):
    if isinstance(denominator, list):
        denom = sum(part[column].fillna(0).sum() for column in denominator)
    else:
        denom = part[denominator].fillna(0).sum()
    if denom <= 0:
        return None
    return round(100 * part[numerator].fillna(0).sum() / denom, 1)


rows = []
for district, part in frame.groupby("Supervisorial"):
    ca11 = part[part["Congressional"] == 11]
    rows.append({
        "supervisor_district": int(district),
        "district_label": f"District {int(district)}",
        "ballots_counted": int(part["ballots"].fillna(0).sum()),
        "registered_voters": int(part["registered"].fillna(0).sum()),
        "turnout_pct": weighted(part, "ballots", "registered"),
        "hilton_votes": int(part["hilton"].fillna(0).sum()),
        "becerra_votes": int(part["becerra"].fillna(0).sum()),
        "hilton_pct": weighted(part, "hilton", ["hilton", "becerra"]),
        "ca11_ballots": int(ca11["ballots"].fillna(0).sum()),
        "chan_votes": int(ca11["chan"].fillna(0).sum()),
        "wiener_votes": int(ca11["wiener"].fillna(0).sum()),
        "chan_pct": weighted(ca11, "chan", ["chan", "wiener"]),
        "measure_b_yes_pct": weighted(part, "measure_b__yes", ["measure_b__yes", "measure_b__no"]),
        "measure_d_yes_pct": weighted(part, "measure_d__yes", ["measure_d__yes", "measure_d__no"]),
        "measure_g_yes_pct": weighted(part, "measure_g__yes", ["measure_g__yes", "measure_g__no"]),
        "measure_i_yes_pct": weighted(part, "measure_i__yes", ["measure_i__yes", "measure_i__no"]),
    })

districts = pd.DataFrame(rows).sort_values("supervisor_district")
districts.to_csv(OUT / "districts.csv", index=False)
print(districts[["supervisor_district", "hilton_pct", "chan_pct", "measure_b_yes_pct"]].to_string(index=False))


In [ ]:
trend = pd.read_csv(SOURCE / "citywide_trend.csv", dtype={"folder": str})
report_dates = {
    "20261103_1": "Nov. 3, first", "20261103_4": "Nov. 3, last", "20261104": "Nov. 4", "20261105": "Nov. 5",
    "20261106": "Nov. 6", "20261107": "Nov. 7", "20261108": "Nov. 8", "20261109": "Nov. 9", "20261110": "Nov. 10",
}
measures = trend[trend["contest"].isin(["Measure B", "Measure I", "Measure D"])]
wide = measures.pivot_table(index=["report", "folder", "ballots_counted"], columns=["contest", "choice"], values="votes")
lines = pd.DataFrame(index=wide.index)
for contest in ["Measure B", "Measure I", "Measure D"]:
    yes, no = wide[(contest, "YES")], wide[(contest, "NO")]
    lines[f"{contest.lower().replace(' ', '_')}_yes_pct"] = (100 * yes / (yes + no)).round(2)
lines = lines.reset_index()
missing = sorted(set(lines["folder"]) - set(report_dates))
if missing:
    raise KeyError(f"Add a date for report folders {missing}")
lines.insert(1, "date_label", [report_dates[f] for f in lines["folder"]])
lines.insert(2, "label", [f"Report {r}, {report_dates[f]}" for r, f in zip(lines["report"], lines["folder"])])
lines = lines.drop(columns="folder")
lines.to_csv(OUT / "trend.csv", index=False)

rows = []
for contest in ["Measure B", "Measure I", "Measure D", "Measure G"]:
    slug = contest.lower().replace(" ", "_")
    yes, no = frame[f"{slug}__yes"].fillna(0), frame[f"{slug}__no"].fillna(0)
    yes_early, no_early = frame[f"{slug}__yes_r5"].fillna(0), frame[f"{slug}__no_r5"].fillna(0)
    late_yes, late_total = (yes - yes_early).sum(), (yes + no - yes_early - no_early).sum()
    rows.append({
        "measure": contest,
        "counted_by_nov_4_yes_pct": round(100 * yes_early.sum() / (yes_early + no_early).sum(), 2),
        "counted_since_yes_pct": round(100 * late_yes / late_total, 2),
    })
late_count = pd.DataFrame(rows)
late_count.to_csv(OUT / "late_count.csv", index=False)
print(lines.to_string(index=False))
print(late_count.to_string(index=False))

In [ ]:
shapes = json.loads((SOURCE / "precincts.geojson").read_text())
for feature in shapes["features"]:
    props = feature["properties"]
    feature["properties"] = {
        "precinct_id": int(props["precinct_id"]),
        "precinct_name": props["VotingPrecinctName"],
        "neighborhood": props["Neighborhood"],
        "supervisor_district": int(props["Supervisorial"]),
        "congressional_district": int(props["Congressional"]),
        "assembly_district": int(props["Assembly"]),
    }
shapes["crs"] = {"type": "name", "properties": {"name": "EPSG:4326"}}
(OUT / "precincts.geojson").write_text(json.dumps(shapes))
print(len(shapes["features"]), "shapes ->", OUT / "precincts.geojson")


In [ ]:
hilton = precincts["hilton_votes"].sum()
becerra = precincts["becerra_votes"].sum()
print("Hilton share", round(100 * hilton / (hilton + becerra), 2))
print("precincts", len(precincts), "with hilton_pct", precincts["hilton_pct"].notna().sum())
print("chan_pct filled", precincts["chan_pct"].notna().sum())
print("blank chan outside CA-11", int((precincts["congressional_district"] != 11).sum()))


## Load these into Datawrapper

For a precinct choropleth, upload `datawrapper/precincts.geojson` as the map and `datawrapper/precincts.csv` as the data. Join on `precinct_id`. Color one of `turnout_pct`, `mail_pct`, `hilton_pct`, `chan_pct`, `measure_b_yes_pct`, `measure_d_yes_pct`, `measure_g_yes_pct`, or `measure_i_yes_pct`.

For a supervisor-district bar chart, upload `datawrapper/districts.csv`. One row per district.

For the late-count line chart, upload `datawrapper/trend.csv` and add a reference line at 50. For the early-versus-late comparison, upload `datawrapper/late_count.csv` as a grouped bar or arrow chart.

`ca11_status` and `neighborhood_name` are for tooltips. In CA-11 maps, show `ca11_status` so the blank CA-15 precincts read "In CA-15" rather than "no data".
